# Method comparison — same experiment, swappable steering method

Shared, minimal notebook so **Charlotte and Arnav compare methods on the SAME experiment** (generation tone).
Everything is fixed and identical **except the one `METHOD` cell (§5)**. Each person sets `METHOD_NAME`,
runs top-to-bottom, saves `method_compare_<name>_<model>.json`; §8 diffs them side by side.

`image_effect` is method-independent (should match — a sanity check). `steer_effect` depends on the METHOD —
that is the comparison: does each steering approach reproduce the image effect?

## 0 · Install

In [ ]:
!pip -q install transformers accelerate pillow numpy vaderSentiment osfclient

## 1 · Config (shared — keep identical across people)

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")   # reduce fragmentation (before torch inits CUDA)
import json, contextlib, math, csv as _csv, glob, zipfile, gc
import numpy as np, torch
from PIL import Image

MODEL   = "google/gemma-3-4b-it"      # SAME light model for both (matches Arnav). Switch to 12B once method is chosen.
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16
OUT_DIR = "/content/out"; os.makedirs(OUT_DIR, exist_ok=True)
N_IMG, N_PROMPT, GEN_LEN, TEMP, IMG_MAXDIM, SEED = 12, 12, 40, 0.0, 512, 0
BATCH   = 8            # batched generation size (raise if GPU has room)
SCORE_AXIS = True      # set False for big fast runs (VADER+RoBERTa carry the comparison; axis costs a forward pass/gen)

# --- affect images: set ONE of these to where yours live ---
AFFECT_DIR = "/content/affect_data"                                          # (b) pre-split subfolders live here
OASIS_BASE = "/content/drive/MyDrive/affect_refusal/oasis"                   # (c) OASIS root (auto-extracts OASIS.zip if needed)
OASIS_CSV  = "/content/drive/MyDrive/affect_refusal/oasis/osfstorage/Data files/OASIS_data.csv"
CANDS = {"lo":["images_negative","negative","distress","lo"],
         "mid":["images_neutral","neutral","mid"],
         "hi":["images_benign_emotional","images_positive","positive","benign_emotional","hi"]}

METHOD_NAME = "charlotte"             # <<< Arnav sets this to "arnav"
print("config ready |", MODEL, "| method:", METHOD_NAME)

## 1a · HF auth

In [ ]:
try:
    from huggingface_hub import login
    _t=os.environ.get("HF_TOKEN")
    if not _t:
        try:
            from google.colab import userdata; _t=userdata.get("HF_TOKEN")
        except Exception: _t=None
    if _t: login(_t); print("HF auth ok")
    else: print("!! add HF_TOKEN in Colab secrets if the model 401s")
except Exception as e: print("auth note:", e)

## 2 · Affect images -> IMGS

Tries in order: (a) reuse `img_lo/mid/hi` already in the kernel, (b) pre-split folders under `AFFECT_DIR`,
(c) OASIS + valence tertiles — **auto-extracting `OASIS.zip`** to fast local disk and matching images to the
CSV by Theme, recursively.

In [ ]:
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print("drive:", e)

def _load(p):
    im=Image.open(p).convert("RGB")
    if max(im.size)>IMG_MAXDIM:
        s=IMG_MAXDIM/max(im.size); im=im.resize((int(im.size[0]*s),int(im.size[1]*s)))
    return im
def _from_split(key):
    for name in CANDS[key]:
        d=os.path.join(AFFECT_DIR,name)
        if os.path.isdir(d):
            fs=[os.path.join(d,f) for f in sorted(os.listdir(d)) if f.lower().endswith((".jpg",".jpeg",".png",".webp"))]
            if fs: return [_load(p) for p in fs[:N_IMG]]
    return None
def _index(root):
    idx={}
    for dp,_,fn in os.walk(root):
        for f in fn:
            if f.lower().endswith((".jpg",".jpeg",".png")):
                idx.setdefault(os.path.splitext(f)[0].strip().lower(), os.path.join(dp,f))
    return idx
def _find_ratings_csv():
    import csv as __c
    cands=[OASIS_CSV]+glob.glob(os.path.join(OASIS_BASE,"**","*.csv"), recursive=True) if os.path.isdir(OASIS_BASE) else [OASIS_CSV]
    for c in cands:
        if not os.path.isfile(c): continue
        try:
            with open(c, encoding="utf-8-sig", errors="ignore", newline="") as f: cols=[x.strip().lower().lstrip("﻿") for x in next(__c.reader(f))]
        except Exception: continue
        if any("valence" in x for x in cols) and any(x in ("theme","item","name","stimulus") for x in cols) and not any(x.startswith("i1") for x in cols):
            return c
    return OASIS_CSV
def _oasis_from_csv():
    global OASIS_CSV
    OASIS_CSV=_find_ratings_csv()
    if not os.path.isfile(OASIS_CSV): return None
    base = OASIS_BASE if os.path.isdir(OASIS_BASE) else os.path.dirname(os.path.dirname(OASIS_CSV))
    idx=_index(base)
    if not idx:                                              # images are zipped -> extract to fast local disk
        zips=glob.glob(os.path.join(base,"**","*.zip"), recursive=True)
        if zips:
            ex="/content/_oasis_imgs"; os.makedirs(ex, exist_ok=True)
            print("extracting", os.path.basename(zips[0]), "-> /content (once per session)...")
            with zipfile.ZipFile(zips[0]) as z: z.extractall(ex)
            idx=_index(ex)
    print("indexed", len(idx), "OASIS images")
    if not idx: return None
    rows=[]
    with open(OASIS_CSV, newline="", encoding="utf-8-sig", errors="ignore") as f:
        for r in _csv.DictReader(f):
            k={(kk or "").strip().lstrip("﻿"):vv for kk,vv in r.items()}
            def g(*ns):
                for n in ns:
                    for kk in k:
                        if kk.lower()==n: return k[kk]
            th=g("theme","item","name","stimulus","file","filename","image"); v=g("valence_mean","valence")
            if not th or v is None: continue
            key=str(th).strip().lower()
            p=idx.get(key) or next((pp for kk2,pp in idx.items() if kk2.startswith(key)), None)
            if p:
                try: rows.append((p, float(v)))
                except: pass
    if len(rows)<30: return None
    rows.sort(key=lambda x:x[1]); t=len(rows)//3
    print(f"OASIS: matched {len(rows)} images -> valence tertiles")
    return dict(lo=[_load(p) for p,_ in rows[:t][:N_IMG]], mid=[_load(p) for p,_ in rows[t:2*t][:N_IMG]], hi=[_load(p) for p,_ in rows[-t:][-N_IMG:]])
def _oasis_from_long():
    import collections
    base = OASIS_BASE if os.path.isdir(OASIS_BASE) else os.path.dirname(os.path.dirname(OASIS_CSV))
    idx=_index(base)
    if not idx:
        zz=glob.glob(os.path.join(base,"**","*.zip"), recursive=True)
        if zz:
            ex="/content/_oasis_imgs"; os.makedirs(ex,exist_ok=True)
            with zipfile.ZipFile(zz[0]) as z: z.extractall(ex)
            idx=_index(ex)
    longs=glob.glob(os.path.join(base,"**","*long*.csv"), recursive=True)
    if not idx or not longs: return None
    sums=collections.defaultdict(lambda:[0.0,0])
    with open(longs[0], encoding="utf-8-sig", errors="ignore", newline="") as f:
        for r in _csv.DictReader(f):
            k={(kk or "").strip().lstrip("﻿").lower():vv for kk,vv in r.items()}
            va=str(k.get("valar","")).strip().lower(); th=k.get("theme"); rt=k.get("rating")
            if th and rt not in (None,"") and va.startswith("val"):
                try: v=float(rt); ss=sums[str(th).strip().lower()]; ss[0]+=v; ss[1]+=1
                except: pass
    vbt={t:s[0]/s[1] for t,s in sums.items() if s[1]>0}
    rows=[]
    for th,v in vbt.items():
        p=idx.get(th) or next((pp for kk,pp in idx.items() if kk.startswith(th)), None)
        if p: rows.append((p,v))
    if len(rows)<30: return None
    rows.sort(key=lambda x:x[1]); t=len(rows)//3
    print(f"OASIS (long): matched {len(rows)} images -> valence tertiles")
    return dict(lo=[_load(p) for p,_ in rows[:t][:N_IMG]], mid=[_load(p) for p,_ in rows[t:2*t][:N_IMG]], hi=[_load(p) for p,_ in rows[-t:][-N_IMG:]])
def load_oasis():
    if all(v in globals() for v in ("img_lo","img_mid","img_hi")):
        return dict(lo=img_lo[:N_IMG], mid=img_mid[:N_IMG], hi=img_hi[:N_IMG])
    lo,mid,hi=_from_split("lo"),_from_split("mid"),_from_split("hi")
    if lo and hi: print("loaded from pre-split folders"); return dict(lo=lo, mid=(mid or lo), hi=hi)
    r=_oasis_from_csv()
    if r: return r
    r=_oasis_from_long()
    if r: return r
    raise FileNotFoundError("No affect images — set OASIS_BASE (OASIS root incl. images/zip) + OASIS_CSV, or AFFECT_DIR pre-split folders.")
IMGS=load_oasis(); print("images:", {k:len(v) for k,v in IMGS.items()})

## 3 · Prompts (shared, neutral)

In [ ]:
PROMPTS=["Write a few sentences about an ordinary afternoon.",
 "Continue this: 'The next morning, she opened the door and'","Describe a walk through a city you have never seen.",
 "Write a short passage about a train arriving at a station.","Continue this: 'He picked up the letter and began to read.'",
 "Describe a room that has been empty for a while.","Write about the view from a window.",
 "Continue this: 'The road stretched on ahead, and'","Describe the inside of an old bookshop.",
 "Write a few sentences about waiting for a bus.","Describe a quiet street at dusk.",
 "Continue this: 'The phone rang twice, and then'"][:N_PROMPT]
print(len(PROMPTS),"prompts")

## 4 · Model + helpers (frees any resident model first; batched generation)

In [ ]:
for _n in ["model","proc","_sent"]:                       # free any model already on the GPU in this kernel
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache(); print("free GB before load:", round(torch.cuda.mem_get_info()[0]/1e9,1))
from transformers import AutoProcessor
try: from transformers import AutoModelForImageTextToText as _AutoVLM
except Exception: from transformers import AutoModelForVision2Seq as _AutoVLM
proc=AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model=_AutoVLM.from_pretrained(MODEL, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True).eval()
tok=proc.tokenizer if hasattr(proc,"tokenizer") else proc
def _layers(m):
    best=None
    for _,mod in m.named_modules():
        if isinstance(mod,torch.nn.ModuleList) and len(mod)>=8 and any(("attn" in n.lower() or "attention" in n.lower()) for n,_ in mod[0].named_modules()): best=mod
    return best
layers=_layers(model); nL=len(layers)
def bi(text, image=None):
    content=([{"type":"image"}] if image is not None else [])+[{"type":"text","text":text}]
    pr=proc.apply_chat_template([{"role":"user","content":content}], add_generation_prompt=True, tokenize=False)
    inp=proc(text=[pr], images=([image] if image is not None else None), return_tensors="pt")
    return {k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in inp.items()}
U=lambda v:(v/v.norm().clamp_min(1e-6)).to(DEVICE,DTYPE)
def add_hook(vec,coef):
    u=U(vec)
    def h(m,i,o): return (o[0]+coef*u,)+tuple(o[1:]) if isinstance(o,tuple) else o+coef*u
    return h
@contextlib.contextmanager
def hk(hooks):
    hd=[layers[l].register_forward_hook(h) for (l,h) in hooks]
    try: yield
    finally:
        for x in hd: x.remove()
def RL_last(inp):
    with torch.no_grad(): out=model(**inp, output_hidden_states=True)
    hs=out.hidden_states[1:1+nL]; return torch.stack([h.float()[0,-1].cpu() for h in hs])
def gen_many(prompts, image=None, hooks=(), bs=None):           # batched; Gemma-3 nested-images format
    bs=bs or BATCH; tok.padding_side="left"; outs=[]
    for i in range(0,len(prompts),bs):
        chunk=prompts[i:i+bs]
        prs=[proc.apply_chat_template([{"role":"user","content":([{"type":"image"}] if image is not None else [])+[{"type":"text","text":p}]}],
                                      add_generation_prompt=True, tokenize=False) for p in chunk]
        imgs=[[image] for _ in chunk] if image is not None else None
        inp=proc(text=prs, images=imgs, return_tensors="pt", padding=True)
        inp={k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in inp.items()}
        L=inp["input_ids"].shape[1]
        kw=dict(max_new_tokens=GEN_LEN, do_sample=(TEMP>0), pad_token_id=tok.eos_token_id)
        if TEMP>0: kw.update(temperature=TEMP, top_p=0.95)
        with torch.no_grad(), hk(hooks): out=model.generate(**inp, **kw)
        outs += [proc.batch_decode(out[j:j+1, L:], skip_special_tokens=True)[0].replace("\n"," ").strip() for j in range(len(chunk))]
    return outs
print("helpers ready | layers", nL)

## 5 · METHOD — the ONLY cell that differs between people

In [ ]:
METHODS = {
 "charlotte": dict(layers="gate", proj="mean", steer="normscaled", alpha=0.008),   # mean-over-layers, norm-scaled, gate [8,20)
 "arnav":     dict(layers="late", proj="last", steer="coeff",      coeff=20.0),     # last-token, fixed coeff-20  (Arnav: edit to your exact layers/coeff)
}
assert "IMGS" in globals(), "Run §2 (data) first — it defines IMGS."
assert "model" in globals() and "RL_last" in globals(), "Run §4 (model + helpers) first."
METHOD=METHODS[METHOD_NAME]
def _mean_last(imgs, prompt="Describe what is happening in this image."):
    return torch.stack([RL_last(bi(prompt,im)) for im in imgs[:N_IMG]]).mean(0)
a_dir=(_mean_last(IMGS["lo"])-_mean_last(IMGS["hi"])); a_dir=a_dir/a_dir.norm(dim=-1,keepdim=True).clamp_min(1e-6)
if   METHOD["layers"]=="gate": LAYERS=[l for l in range(8,20) if l<nL]
elif METHOD["layers"]=="late": LAYERS=list(range(int(0.7*nL), nL))
else: LAYERS=[l for l in METHOD["layers"] if l<nL]
with torch.no_grad(): _o=model(**bi(PROMPTS[0]), output_hidden_states=True)
norms=np.array([float(h[0,-1].float().norm()) for h in _o.hidden_states[1:1+nL]])
def steer_hooks(sign):     # +1 = toward NEGATIVE valence (+a); -1 = toward POSITIVE (-a)
    if METHOD["steer"]=="normscaled": return [(l, add_hook(a_dir[l], sign*METHOD["alpha"]*norms[l])) for l in LAYERS]
    return [(l, add_hook(a_dir[l], sign*float(METHOD["coeff"]))) for l in LAYERS]
print("METHOD:", METHOD_NAME, METHOD, "| layers", LAYERS[:3], "...", LAYERS[-1])

## 6 · Scorers (shared)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_vader=SentimentIntensityAnalyzer()
def score_vader(t): return float(_vader.polarity_scores(t or ".")["compound"])
try:
    from transformers import pipeline
    _sent=pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", top_k=None, device=0 if DEVICE=="cuda" else -1)
    def score_roberta(t):
        d={x["label"].lower():x["score"] for x in _sent((t or ".")[:512])[0]}; return float(d.get("positive",0)-d.get("negative",0))
except Exception as e:
    score_roberta=None; print("RoBERTa unavailable:", e)
def score_axis(t):
    if not t.strip(): return 0.0
    with torch.no_grad(): o=model(**bi(t), output_hidden_states=True)
    hs=o.hidden_states[1:1+nL]
    r=[(hs[l][0].float().mean(0) if METHOD["proj"]=="mean" else hs[l][0,-1].float()).cpu() for l in range(nL)]
    return -float(np.mean([float(r[l]@a_dir[l]) for l in range(nL)]))/1000.0
def score(t): return (score_vader(t), score_roberta(t) if score_roberta else float("nan"), score_axis(t) if SCORE_AXIS else float("nan"))
print("scorers ready | SCORE_AXIS =", SCORE_AXIS)

## 7 · Run + save (per method)

In [ ]:
def cond(label, hooks=(), imgs=None):
    A=[]
    if imgs is None:
        A += [score(t) for t in gen_many(PROMPTS, hooks=hooks)]
    else:
        for im in imgs[:N_IMG]:
            A += [score(t) for t in gen_many(PROMPTS, image=im)]
    return dict(label=label, n=len(A), raw=np.array(A))
C={}
for lab,kw in [("image_distress",dict(imgs=IMGS["lo"])),("image_positive",dict(imgs=IMGS["hi"])),
               ("steer_+a_neg",dict(hooks=steer_hooks(+1))),("steer_-a_pos",dict(hooks=steer_hooks(-1)))]:
    print("running", lab, "..."); C[lab]=cond(lab, **kw)
def boot(a,b,col,it=2000):
    rng=np.random.default_rng(SEED); da=a[:,col][~np.isnan(a[:,col])]; db=b[:,col][~np.isnan(b[:,col])]
    d=float(np.mean(da)-np.mean(db)); bs=[np.mean(rng.choice(da,len(da)))-np.mean(rng.choice(db,len(db))) for _ in range(it)]
    return d, float(np.percentile(bs,2.5)), float(np.percentile(bs,97.5))
EFF={}
for nm,hi,lo in [("image_effect","image_positive","image_distress"),("steer_effect","steer_-a_pos","steer_+a_neg")]:
    EFF[nm]={s:dict(zip(("diff","ci_lo","ci_hi"), boot(C[hi]["raw"],C[lo]["raw"],col))) for col,s in [(0,"VADER"),(1,"RoBERTa"),(2,"axis")]}
print("\n%-13s %9s %9s %9s"%("effect","VADER","RoBERTa","axis"))
for nm in EFF: print("%-13s %+9.3f %+9.3f %+9.3f"%(nm, EFF[nm]["VADER"]["diff"], EFF[nm]["RoBERTa"]["diff"], EFF[nm]["axis"]["diff"]))
out=dict(method=METHOD_NAME, method_params=METHOD, model=MODEL, layers=LAYERS, effects=EFF)
tag="%s_%s"%(METHOD_NAME, MODEL.split("/")[-1]); json.dump(out, open(f"{OUT_DIR}/method_compare_{tag}.json","w"), indent=2, default=float)
print("\nsaved -> method_compare_%s.json  (now run the OTHER METHOD_NAME, then §8)"%tag)

## 8 · Compare — side by side (after both methods saved)

In [ ]:
import glob
runs=[json.load(open(f)) for f in sorted(glob.glob(f"{OUT_DIR}/method_compare_*.json"))]
if len(runs)<2:
    print("Only %d method run(s). Run the notebook with each METHOD_NAME, then re-run this."%len(runs))
else:
    print("%-13s %-9s "%("","")+" ".join("%-24s"%r["method"] for r in runs))
    for eff in ("image_effect","steer_effect"):
        for sc in ("VADER","RoBERTa","axis"):
            row="%-13s %-9s"%(eff if sc=="VADER" else "", sc)
            for r in runs:
                e=r["effects"][eff][sc]; sig="*" if (e["ci_lo"]>0 or e["ci_hi"]<0) else " "
                row+=" %+7.3f[%+.2f,%+.2f]%s"%(e["diff"],e["ci_lo"],e["ci_hi"],sig)
            print(row)
    print("\nimage_effect is method-independent -> should match (sanity). steer_effect is THE comparison.")

## 9 · Comparison figure

In [ ]:
import glob, json
import numpy as np, matplotlib.pyplot as plt
runs={json.load(open(f))["method"]:json.load(open(f)) for f in sorted(glob.glob(f"{OUT_DIR}/method_compare_*.json"))}
methods=list(runs); scorers=["VADER","RoBERTa","axis"]
effects=[("image_effect","Image effect (positive - distress)"),("steer_effect","Steer effect (+valence - -valence)")]
colors=dict(zip(methods,["#0072B2","#E69F00","#009E73","#CC79A7"]))
fig,axs=plt.subplots(1,2,figsize=(11,4.2))
for ax,(ekey,etitle) in zip(axs,effects):
    x=np.arange(len(scorers)); w=0.8/max(1,len(methods))
    for mi,m in enumerate(methods):
        e=runs[m]["effects"][ekey]; vals=[e[sc]["diff"] for sc in scorers]
        lo=[e[sc]["diff"]-e[sc]["ci_lo"] for sc in scorers]; hi=[e[sc]["ci_hi"]-e[sc]["diff"] for sc in scorers]
        ax.bar(x+mi*w-0.4+w/2, vals, w, yerr=[lo,hi], capsize=3, label=m, color=colors.get(m))
    ax.axhline(0,color="#888",lw=1); ax.set_xticks(x); ax.set_xticklabels(scorers)
    ax.set_title(etitle,fontsize=11); ax.set_ylabel("valence effect")
axs[0].legend(title="method")
fig.suptitle("Method comparison - image effect matches; steer effect is where methods differ",fontweight="bold")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_method_compare.png",dpi=200,bbox_inches="tight"); plt.show()
print("saved fig_method_compare.png")

## 10 · Interpretation

In [ ]:
import glob, json
runs={json.load(open(f))["method"]:json.load(open(f)) for f in sorted(glob.glob(f"{OUT_DIR}/method_compare_*.json"))}
ms=list(runs)
def sig(e): return e["ci_lo"]>0 or e["ci_hi"]<0
print("METHOD COMPARISON - generation tone\n"+"="*46)
if len(ms)>=2:
    a,b=ms[0],ms[1]
    for sc in ("VADER","RoBERTa"):
        da=runs[a]["effects"]["image_effect"][sc]["diff"]; db=runs[b]["effects"]["image_effect"][sc]["diff"]
        print(f"SANITY image_effect {sc:8s}: {a} {da:+.3f} vs {b} {db:+.3f} -> {'MATCH' if abs(da-db)<0.05 else 'DIFFER (check shared data/prompts!)'}")
    print("  (axis image_effect differs by design - each method uses its own projection convention.)")
print("\nSTEER EFFECT (the comparison; * = 95% CI excludes 0):")
for m in ms:
    e=runs[m]["effects"]["steer_effect"]
    row=" | ".join(f"{sc} {e[sc]['diff']:+.2f}{'*' if sig(e[sc]) else ' '}" for sc in ("VADER","RoBERTa","axis"))
    moves=sig(e["VADER"]) or sig(e["RoBERTa"])
    print(f"  {m:10s} {row}   -> {'MOVES generation tone' if moves else 'NO detectable tone effect'}")
movers=[m for m in ms if sig(runs[m]["effects"]["steer_effect"]["VADER"]) or sig(runs[m]["effects"]["steer_effect"]["RoBERTa"])]
inert=[m for m in ms if m not in movers]
print("\nVERDICT:")
if movers: print("  Steering reproduces the tone effect for:", ", ".join(movers))
if inert:  print("  No detectable steer effect for:", ", ".join(inert), "\n  -> likely a magnitude mismatch (fixed coefficient << norm-scaled) or layer placement (late vs mid-gate).")
print("\nTakeaway: image_effect is shared (same experiment); the method whose steer moves tone reproduces the behavioral effect.")

## 11 - Fair comparison (magnitude / layer sweep)

In [ ]:

import numpy as np, matplotlib.pyplot as plt
GATE=[l for l in range(8,20) if l<nL]; LATE=list(range(int(0.7*nL), nL))
gate_norm=float(np.mean([norms[l] for l in GATE]))
def steer_eff(hooks_fn):
    pos=np.mean([score_vader(t) for t in gen_many(PROMPTS, hooks=hooks_fn(-1))])
    neg=np.mean([score_vader(t) for t in gen_many(PROMPTS, hooks=hooks_fn(+1))])
    return float(pos-neg)
series={}
for a in [0.004,0.008,0.016]:
    series.setdefault("norm-scaled @ gate (Charlotte)",[]).append((a*gate_norm, steer_eff(lambda s,a=a:[(l,add_hook(a_dir[l],s*a*norms[l])) for l in GATE])))
for c in [20,80,320,1280]:
    series.setdefault("fixed-coeff @ late (Arnav)",[]).append((c, steer_eff(lambda s,c=c:[(l,add_hook(a_dir[l],s*c)) for l in LATE])))
for c in [20,80,320]:
    series.setdefault("fixed-coeff @ gate",[]).append((c, steer_eff(lambda s,c=c:[(l,add_hook(a_dir[l],s*c)) for l in GATE])))
plt.figure(figsize=(7.5,4.6))
for name,pts in series.items():
    xs,ys=zip(*pts); plt.plot(xs,ys,"-o",label=name)
    for x,y in pts: print(f"{name:32s} mag {x:7.0f} -> steer_effect {y:+.2f}")
plt.axhline(0,color="#888",lw=1); plt.xscale("log")
plt.xlabel("effective per-layer steering magnitude (log)"); plt.ylabel("steer_effect (VADER)")
plt.title("Fair comparison - does the effect appear at matched magnitude / mid-layers?")
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_fair_compare.png",dpi=180,bbox_inches="tight"); plt.show()
